# Đánh giá prompt bằng số đo trên UIT-VSFC

"Prompt này tốt hơn" chỉ có nghĩa khi có số. Bạn so bốn cách hỏi `qwen2.5:3b` cùng một việc (phân loại cảm xúc phản
hồi sinh viên): **zero-shot**, **few-shot**, **CoT-lite** (một câu suy luận rồi mới ra nhãn) và **JSON schema**
(Ollama ép output theo schema), trên **150 câu ghim** của tập test, với macro-F1, khoảng tin cậy bootstrap, số token
và độ trễ. Rồi đo **sàn nhiễu**: cùng prompt, chạy lại, bao nhiêu câu đổi nhãn.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, hằng. Điền vào chỗ `# TODO`. Giữ nguyên các hằng prompt: test và số liệu lý
  thuyết dựa trên đúng chúng.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu, gọi Ollama, in bảng; test bỏ qua chúng.
- Cần Ollama với `qwen2.5:3b`. Câu trả lời được cache vào `data/raw/uit-vsfc/cache/`, chạy lại không gọi model nữa.
- Chấm: `uv run pytest tracks/ai-engineer/modules/11-llm-apis-prompting/exercises/intermediate/02-prompt-eval -q`

In [ ]:
# explore: tìm gốc repo, tải UIT-VSFC nếu chưa có
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

VSFC = ensure("uit-vsfc")
CACHE = VSFC / "cache"
CACHE.mkdir(exist_ok=True)
print(VSFC, sorted(p.name for p in (VSFC / "default").iterdir()))

In [ ]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")

import json
import re
import time
from collections import Counter

import numpy as np

## 1. Bốn biến thể prompt

Các hằng dưới đây là prompt dùng ở mục 4 và 7 lý thuyết; **đừng sửa** (muốn thử prompt của bạn thì thêm biến thể mới
ở cuối notebook). Nhãn viết bằng tiếng Anh (`negative`, `neutral`, `positive`) để đọc ra bằng code chắc chắn hơn:
"tích cực" là chuỗi con của "không tích cực".

`build_messages(variant, sentence, examples) -> list[dict]`, với `user = "Phản hồi: <sentence>"`:

| variant | messages |
| --- | --- |
| `zero_shot` | `system` = `TASK + ONE_WORD`, rồi `user` |
| `few_shot` | `system` = `TASK + ONE_WORD`; với mỗi `(câu, nhãn)` trong `examples` theo thứ tự: `user` "Phản hồi: <câu>" rồi `assistant` "<nhãn>"; cuối cùng `user` |
| `cot` | `system` = `TASK + COT`, rồi `user` |
| `json_schema` | `system` = `TASK + JSON_INSTR`, rồi `user` (schema gửi riêng qua `format`, không nằm trong prompt) |

Biến thể khác: `ValueError`. `examples` chỉ dùng cho `few_shot`.

`parse_label(variant, reply) -> str | None`:

- `zero_shot`, `few_shot`: lấy mọi từ chữ cái Latinh của `reply.lower()`; nếu các từ là nhãn chỉ gồm **đúng một** nhãn
  khác nhau (có thể lặp) thì trả nhãn đó, còn lại (không có nhãn, hay có hai nhãn khác nhau) trả `None`.
- `cot`: nhãn đứng sau lần **cuối cùng** của `Nhãn:` (không phân biệt hoa thường; giữa `:` và nhãn có thể có khoảng
  trắng, `*`, dấu backtick). Không có thì `None`. Nhãn nhắc tới trong câu suy luận không tính.
- `json_schema`: `json.loads` cả câu trả lời (sau `strip`); phải là object có `label` thuộc `LABELS`; còn lại `None`.

In [ ]:
LABELS = ["negative", "neutral", "positive"]
VARIANTS = ["zero_shot", "few_shot", "cot", "json_schema"]

TASK = (
    "Bạn phân loại cảm xúc của một câu phản hồi của sinh viên về môn học, giảng viên hoặc cơ sở vật chất.\n"
    "Nhãn:\n"
    "- negative: chê, phàn nàn, hoặc góp ý chỉ ra điều chưa tốt.\n"
    "- neutral: không khen không chê, chỉ mô tả, hỏi, cảm ơn chung chung, hoặc không có ý kiến.\n"
    "- positive: khen, hài lòng.\n"
)
ONE_WORD = "Chỉ trả lời đúng một từ: negative, neutral hoặc positive."
COT = ("Trả lời đúng hai dòng. Dòng 1 bắt đầu bằng `Ý chính:` rồi một câu ngắn của bạn nói phản hồi đang khen, "
       "chê, góp ý hay trung lập, vì sao. Dòng 2 bắt đầu bằng `Nhãn:` rồi đúng một trong ba nhãn.")
JSON_INSTR = "Trả lời bằng JSON có đúng một khoá `label`."
SCHEMA = {"type": "object", "properties": {"label": {"type": "string", "enum": LABELS}},
          "required": ["label"], "additionalProperties": False}
NUM_PREDICT = {"zero_shot": 8, "few_shot": 8, "cot": 160, "json_schema": 32}

In [ ]:
def build_messages(variant: str, sentence: str, examples: list[tuple[str, str]]) -> list[dict]:
    """Messages cho một biến thể (bảng ở trên). ValueError với biến thể lạ."""
    # TODO
    raise NotImplementedError


def parse_label(variant: str, reply: str) -> str | None:
    """Nhãn đọc được từ câu trả lời theo quy tắc của biến thể, hoặc None."""
    # TODO
    raise NotImplementedError

## 2. Macro-F1 và bootstrap theo cặp

`macro_f1(y_true, y_pred) -> float`: trung bình F1 của ba nhãn trong `LABELS`. Dự đoán `None` (không đọc được) là
**sai**: nó là false negative của nhãn đúng, không là false positive của nhãn nào. Nhãn không có true positive nào thì
F1 = 0. Kết quả phải khớp `sklearn.metrics.f1_score(..., labels=LABELS, average="macro", zero_division=0)`.

`paired_bootstrap(y_true, pred_a, pred_b, n_boot=2000, seed=0) -> (diff, lo, hi)`: `diff` là macro-F1 của B trừ của A
trên cả tập; `lo`, `hi` là phân vị 2.5 và 97.5 của hiệu đó qua `n_boot` lần lấy mẫu lại **cùng chỉ số cho cả A và
B** (theo cặp). Để số của bạn trùng số của lý thuyết và test, làm đúng thứ tự này: `rng = np.random.default_rng(seed)`,
mỗi vòng `idx = rng.integers(0, n, n)`, hiệu của vòng là `macro_f1(y[idx], b[idx]) - macro_f1(y[idx], a[idx])`, cuối
cùng `np.percentile(hiệu, [2.5, 97.5])`. Trả ba số `float`.

In [ ]:
def macro_f1(y_true: list[str], y_pred: list[str | None]) -> float:
    """Macro-F1 trên LABELS; None là dự đoán sai."""
    # TODO
    raise NotImplementedError


def paired_bootstrap(y_true, pred_a, pred_b, n_boot: int = 2000, seed: int = 0) -> tuple[float, float, float]:
    """(macro-F1(B) - macro-F1(A), phân vị 2.5, phân vị 97.5) qua bootstrap theo cặp."""
    # TODO
    raise NotImplementedError

## 3. Chạy bốn biến thể trên 150 câu ghim

Tập ghim: tập **test**, mỗi nhãn 50 câu lấy bằng `sample(50, random_state=0)`, trộn bằng `random_state=0`. Cân bằng
nhãn có chủ đích: tập test thật chỉ có 5% neutral, macro-F1 trên tập lệch bị hai lớp lớn chi phối (module 04 mục 5).
Ví dụ few-shot: 2 câu mỗi nhãn từ tập **train** (4 tới 20 từ, `random_state=0`), xếp xen kẽ negative, neutral,
positive.

Lời gọi đi thẳng vào REST `/api/chat` của Ollama (không qua `llmkit.chat`) để lấy `prompt_eval_count`, `eval_count`
và đặt `num_predict`, `seed`. Mỗi biến thể khoảng 150 lời gọi; trên PC của track (GPU dùng chung) cả bốn biến thể mất
khoảng 8 phút, phần lớn là CoT.

In [ ]:
# explore: tập ghim 150 câu và 6 ví dụ few-shot
import pandas as pd

test = pd.read_parquet(VSFC / "default" / "test" / "0000.parquet")
parts = [test[test.sentiment == c].sample(50, random_state=0) for c in range(3)]
pinned = pd.concat(parts).sample(frac=1.0, random_state=0).reset_index().rename(columns={"index": "row"})
gold = [LABELS[s] for s in pinned.sentiment]

train = pd.read_parquet(VSFC / "default" / "train" / "0000.parquet")
train = train[train.sentence.str.split().str.len().between(4, 20)]
picked = [train[train.sentiment == c].sample(2, random_state=0).to_dict("records") for c in range(3)]
EXAMPLES = [(picked[c][i]["sentence"], LABELS[c]) for i in range(2) for c in range(3)]
print(len(pinned), Counter(gold))
for text, label in EXAMPLES:
    print(f"{label:9s} {text}")

In [ ]:
# explore: gọi Ollama cho từng biến thể, cache theo (nhãn lượt chạy, câu)
import httpx

CACHE_FILE = CACHE / "m11-prompt-eval.json"
cache = json.loads(CACHE_FILE.read_text(encoding="utf-8")) if CACHE_FILE.exists() else {}


def call(variant: str, sentence: str) -> dict:
    payload = {"model": "qwen2.5:3b", "messages": build_messages(variant, sentence, EXAMPLES), "stream": False,
               "options": {"temperature": 0, "seed": 0, "num_predict": NUM_PREDICT[variant]}}
    if variant == "json_schema":
        payload["format"] = SCHEMA
    t0 = time.perf_counter()
    r = httpx.post("http://localhost:11434/api/chat", json=payload, timeout=300).json()
    return {"reply": r["message"]["content"], "seconds": time.perf_counter() - t0 - r.get("load_duration", 0) / 1e9,
            "prompt_tokens": r.get("prompt_eval_count", 0), "output_tokens": r.get("eval_count", 0),
            "done_reason": r.get("done_reason")}


def run(variant: str, tag: str | None = None) -> list[dict]:
    tag = tag or variant
    out = []
    for row, sentence in zip(pinned.row, pinned.sentence):
        key = f"{tag}|{row}"
        if key not in cache:
            cache[key] = call(variant, sentence)
            CACHE_FILE.write_text(json.dumps(cache, ensure_ascii=False), encoding="utf-8")
        out.append(cache[key])
    return out


results = {v: run(v) for v in VARIANTS}
preds = {v: [parse_label(v, r["reply"]) for r in results[v]] for v in VARIANTS}

print(f"{'biến thể':12s} {'macro-F1':>8s} {'CI 95% (bootstrap)':>20s} {'acc':>6s} {'None':>5s} "
      f"{'p50 s':>6s} {'token vào':>9s} {'token ra':>8s}")
rng_ci = {}
for v in VARIANTS:
    p = preds[v]
    _, lo, hi = paired_bootstrap(gold, [None] * len(gold), p)
    acc = np.mean([g == q for g, q in zip(gold, p)])
    secs = np.median([r["seconds"] for r in results[v]])
    tin = np.mean([r["prompt_tokens"] for r in results[v]])
    tout = np.mean([r["output_tokens"] for r in results[v]])
    print(f"{v:12s} {macro_f1(gold, p):8.3f} {f'[{lo:.3f}, {hi:.3f}]':>20s} {acc:6.3f} {p.count(None):5d} "
          f"{secs:6.2f} {tin:9.0f} {tout:8.1f}")

In [ ]:
# explore: so từng biến thể với zero-shot (bootstrap theo cặp) và đo sàn nhiễu: chạy lại zero-shot y hệt
for v in VARIANTS[1:]:
    diff, lo, hi = paired_bootstrap(gold, preds["zero_shot"], preds[v])
    changed = sum(a != b for a, b in zip(preds["zero_shot"], preds[v]))
    print(f"{v:12s} - zero_shot: {diff:+.3f}  [{lo:+.3f}, {hi:+.3f}]  câu đổi nhãn: {changed}")

repeat = [parse_label("zero_shot", r["reply"]) for r in run("zero_shot", "zero_shot_lan2")]
flips = sum(a != b for a, b in zip(preds["zero_shot"], repeat))
print(f"\nzero-shot chạy lại: {flips} / {len(gold)} câu đổi nhãn, macro-F1 {macro_f1(gold, repeat):.3f}")

In [ ]:
# explore: ma trận nhầm lẫn và vài câu neutral bị đoán sai
for v in VARIANTS:
    m = Counter(zip(gold, preds[v]))
    print(v)
    print("   thật \\ đoán  " + "  ".join(f"{p:>8s}" for p in LABELS + ["None"]))
    for g in LABELS:
        print(f"   {g:12s}" + "  ".join(f"{m[(g, p)]:8d}" for p in LABELS + [None]))

wrong = [(s, preds["few_shot"][i]) for i, (s, g) in enumerate(zip(pinned.sentence, gold))
         if g == "neutral" and preds["few_shot"][i] != "neutral"]
print(f"\nfew-shot đoán sai {len(wrong)} câu neutral, ví dụ:")
for s, p in wrong[:6]:
    print(f"  -> {p:9s} {s}")
print("\nCoT, ví dụ câu trả lời:", repr(results["cot"][0]["reply"]))

In [ ]:
# explore: (tuỳ chọn) so với một model API qua gateway tương thích OpenAI. Mặc định TẮT.
# Đặt RUN_API = True khi đã khai báo OMNIROUTE_BASE_URL, OMNIROUTE_API_KEY, OMNIROUTE_MODEL trong setup/.env.
# Chỉ gửi N_API câu (mỗi câu một lời gọi tính tiền / tính hạn mức): đủ để nhìn, KHÔNG đủ để kết luận.
# Gateway dùng chung có thể khoá model sau vài lần timeout: timeout ngắn, lỗi một lần là dừng, không gọi lại.
RUN_API = False
N_API = 20
if RUN_API:
    import llmkit
    from testkit import _load_env_file

    _load_env_file(ROOT / "setup" / ".env")
    api = llmkit.OpenAICompatClient(os.environ["OMNIROUTE_BASE_URL"], os.environ["OMNIROUTE_API_KEY"],
                                    model=os.environ["OMNIROUTE_MODEL"], timeout=60)
    api_pred = []
    for sentence in pinned.sentence[:N_API]:
        try:
            reply = api.chat(build_messages("json_schema", sentence, EXAMPLES), json_schema=SCHEMA)
        except llmkit.LLMError as exc:
            print("gateway lỗi, dừng (không gọi lại):", exc)
            break
        api_pred.append(parse_label("json_schema", reply))
    n = len(api_pred)
    if n:
        local = preds["json_schema"][:n]
        print(f"{n} câu đầu: API đúng {sum(a == g for a, g in zip(api_pred, gold))}, "
              f"qwen2.5:3b đúng {sum(a == g for a, g in zip(local, gold))}")

## Đọc kết quả

So với bảng ở mục 7 lý thuyết (cùng prompt, cùng tập ghim). Tự trả lời:

- Biến thể nào hơn zero-shot với khoảng tin cậy **không chứa 0**? Chênh lệch nào nằm trong sàn nhiễu (số câu đổi
  nhãn khi chạy lại y hệt)?
- Few-shot đổi phân phối nhãn đoán ra thế nào so với zero-shot? Đọc 6 câu neutral bị đoán sai và 6 ví dụ few-shot:
  ví dụ neutral ("cho em xin ít con chip nfc", "không ý kiến thêm") dạy model điều gì về lớp neutral?
- CoT tốn bao nhiêu token ra và bao nhiêu giây mỗi câu so với zero-shot? Với 1 000 câu mỗi ngày qua API, chênh lệch
  chi phí là bao nhiêu (dùng `workload_cost` của bài Basic 02 với bảng giá ở mục 3 lý thuyết)?
- Nhãn "đúng" của UIT-VSFC có sai không? Đọc vài câu mà cả bốn biến thể cùng "sai" (ví dụ "em xin cảm ơn thầy" được
  gán neutral).

Sau đó làm phần trắc nghiệm của bài trong app.